# Lab 9: Cats vs Dogs Image Recognition using CNN

This notebook builds a Convolutional Neural Network (CNN) with strong layered design (BatchNorm + Dropout + Data Augmentation) for classifying cats and dogs.

In [ ]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

In [ ]:
# Download cats and dogs dataset
_URL = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'
path_to_zip = keras.utils.get_file('cats_and_dogs_filtered.zip', origin=_URL, extract=True)
dataset_dir = pathlib.Path(path_to_zip).with_suffix('')
train_dir = dataset_dir / 'train'
validation_dir = dataset_dir / 'validation'

print('Train dir:', train_dir)
print('Validation dir:', validation_dir)

In [ ]:
# Data pipeline
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
SEED = 123

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    validation_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print('Classes:', class_names)

In [ ]:
# Performance optimization
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
# Data augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='data_augmentation')

for images, labels in train_ds.take(1):
    plt.figure(figsize=(8, 8))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        aug_img = data_augmentation(images)[i].numpy().astype('uint8')
        plt.imshow(aug_img)
        plt.title(class_names[labels[i]])
        plt.axis('off')
    plt.show()

In [ ]:
# CNN model with strong layered architecture
model = keras.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(256, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Training
EPOCHS = 15
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
# Evaluation
loss, acc = model.evaluate(val_ds)
print(f'Validation Loss: {loss:.4f}')
print(f'Validation Accuracy: {acc:.4f}')

In [ ]:
# Plot training history
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Accuracy')
plt.plot(epochs_range, val_acc, label='Val Accuracy')
plt.legend(loc='lower right')
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.legend(loc='upper right')
plt.title('Loss')

plt.show()

In [ ]:
# Predict on a single image
def predict_image(image_path):
    img = keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    prob = model.predict(img_array, verbose=0)[0][0]
    pred_class = class_names[1] if prob >= 0.5 else class_names[0]

    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Prediction: {pred_class} (confidence: {max(prob, 1-prob):.2%})')
    plt.show()

# Example (update path to your local test image):
# predict_image('/path/to/cat_or_dog.jpg')